# Aggregating data

In [1]:
import pandas as pd
import numpy as np

## load dfs

In [2]:
coffee = pd.read_csv('warmup-data/coffee-new.csv')
coffee.head()

,Day,Coffee Type,Units Sold,new-price,revenue
0,Monday,Espresso,40.0,3.99,99.75
1,Monday,Latte,40.0,5.99,89.85
2,Wednesday,Espresso,NaN,3.99,139.65
3,Wednesday,Latte,NaN,5.99,149.75
4,Thursday,Espresso,40.0,3.99,159.60


In [3]:
bios = pd.read_csv('data/bios.csv')
bios.head()

,athlete_id,name,born_date,born_city,born_region,born_country,NOC,height_cm,weight_kg,died_date
0,1,Jean-François Blanchy,1886-12-12,Bordeaux,Gironde,FRA,France,NaN,NaN,1960-10-02
1,2,Arnaud Boetsch,1969-04-01,Meulan,Yvelines,FRA,France,183.0,76.0,NaN
2,3,Jean Borotra,1898-08-13,Biarritz,Pyrénées-Atlantiques,FRA,France,183.0,76.0,1994-07-17
3,4,Jacques Brugnon,1895-05-11,Paris VIIIe,Paris,FRA,France,168.0,64.0,1978-03-20
4,5,Albert Canet,1878-04-17,Wandsworth,England,GBR,France,NaN,NaN,1930-07-25


## value_counts()

In [4]:
bios.value_counts().head()

athlete_id  name               born_date   born_city       born_region             born_country  NOC           height_cm  weight_kg  died_date 
3           Jean Borotra       1898-08-13  Biarritz        Pyrénées-Atlantiques    FRA           France        183.0      76.0       1994-07-17    1
70111       Erna Steinberg     1911-06-30  Charlottenburg  Berlin                  GER           Germany       168.0      54.0       2001-04-21    1
70058       Christa Merten     1944-10-14  Dobbertin       Mecklenburg-Vorpommern  GER           West Germany  168.0      53.0       1986-07-01    1
70062       Paula Mollenhauer  1908-12-22  Hamburg         Hamburg                 GER           Germany       175.0      82.0       1988-07-07    1
70068       Helma Notte        1911-09-22  Düsseldorf      Nordrhein-Westfalen     GER           Germany       175.0      64.0       1997-03-14    1
Name: count, dtype: int64

### using subset=

In [5]:
bios.value_counts(subset='born_city').head()

born_city
Budapest           1378
Moskva (Moscow)     883
Oslo                708
Stockholm           629
Praha (Prague)      600
Name: count, dtype: int64

### using df[col].value_counts()

In [6]:
bios['born_city'].value_counts().head()

born_city
Budapest           1378
Moskva (Moscow)     883
Oslo                708
Stockholm           629
Praha (Prague)      600
Name: count, dtype: int64

### filter multiple fields

In [7]:
# COUNTRY = 'USA'
COUNTRY = 'ITA'
COUNTRIES = ['ITA', 'USA']
LINE = '----------'
SEPARATOR = pd.Series([LINE], index=[LINE])

# CONDITON = bios['born_country'].isin(COUNTRIES)
CONDITION = bios['born_country'] == COUNTRY
# display(CONDITON.head())

region_count = bios[CONDITION]['born_region'].value_counts()
pd.concat([
    region_count.head(),
    SEPARATOR,
    region_count.tail(),
])


Roma                    413
Milano                  367
Bolzano-Bozen           330
Napoli                  179
Torino                  178
----------       ----------
Nuoro                     3
Enna                      2
Potenza                   2
Ragusa                    2
Matera                    1
dtype: object

## groupby

In [8]:
COUNTRY = 'ITA'
CONDITION = bios['born_country'] == COUNTRY
# this is wrong, because we're getting a series, not a df
# bios[CONDITION].groupby('born_region')['athlete_id'].count().sort_values(by='athlete_id', ascending=False)

# group by born region, count athletes within the region, sort the results desceding
bios[CONDITION].groupby('born_region')['athlete_id'].count().sort_values(ascending=False).head()

born_region
Roma             413
Milano           367
Bolzano-Bozen    330
Napoli           179
Torino           178
Name: athlete_id, dtype: int64

In [9]:
# group by coffee type, compute the sum of units sold within each group
coffee.groupby('Coffee Type')['Units Sold'].sum()

Coffee Type
Espresso    215.0
Latte       175.0
Name: Units Sold, dtype: float64

## agg

In [28]:
coffee.rename(columns={'new-price': 'price'}, inplace=True)

In [30]:
coffee.groupby(
    'Coffee Type'
).agg({
    'Units Sold':'sum',
    'price': 'mean'
})

,Units Sold,price
Coffee Type,,
Espresso,215.0,3.99
Latte,175.0,5.99


## pivot

In [36]:
display(coffee.head())
# Define the custom order for days
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
coffee['Day'] = pd.Categorical(coffee['Day'], categories=day_order, ordered=True)

display(coffee.head())
# Create the pivot table
coffee.pivot(
    columns='Coffee Type',
    index='Day',
    # values='revenue'
)



,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,40.0,3.99,99.75
1,Monday,Latte,40.0,5.99,89.85
2,Wednesday,Espresso,NaN,3.99,139.65
3,Wednesday,Latte,NaN,5.99,149.75
4,Thursday,Espresso,40.0,3.99,159.60


,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,40.0,3.99,99.75
1,Monday,Latte,40.0,5.99,89.85
2,Wednesday,Espresso,NaN,3.99,139.65
3,Wednesday,Latte,NaN,5.99,149.75
4,Thursday,Espresso,40.0,3.99,159.60


Units Sold          price        revenue        
Coffee Type   Espresso Latte Espresso Latte Espresso   Latte
Day                                                         
Monday            40.0  40.0     3.99  5.99    99.75   89.85
Wednesday          NaN   NaN     3.99  5.99   139.65  149.75
Thursday          40.0  30.0     3.99  5.99   159.60  179.70
Friday            45.0  35.0     3.99  5.99   179.55  209.65
Saturday          45.0  35.0     3.99  5.99   179.55  209.65
Sunday            45.0  35.0     3.99  5.99   179.55  209.65

---

In [18]:
###############################

### *experiments*
#### cannot use == between series

In [10]:
try:
    assert (
        bios.value_counts(subset='born_city').head() 
        == bios['born_city'].value_counts().head()
    )
except ValueError as ve:
    print(repr(ve))

ValueError('The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().')


#### using .equals()

In [11]:
assert (
    bios.value_counts(subset='born_city').head().equals(
        bios['born_city'].value_counts().head()
    )
)


In [12]:
bios.value_counts().head(2)     # df.valuecounts() aggregates entire rows

athlete_id  name            born_date   born_city       born_region           born_country  NOC      height_cm  weight_kg  died_date 
3           Jean Borotra    1898-08-13  Biarritz        Pyrénées-Atlantiques  FRA           France   183.0      76.0       1994-07-17    1
70111       Erna Steinberg  1911-06-30  Charlottenburg  Berlin                GER           Germany  168.0      54.0       2001-04-21    1
Name: count, dtype: int64

In [13]:
bios['athlete_id'].value_counts().head(3)   # ids should be all different

athlete_id
1        1
97783    1
97777    1
Name: count, dtype: int64

In [14]:
bios['born_country'].value_counts().head(3)     # count of atheletes by country

born_country
USA    9641
GER    6891
GBR    5792
Name: count, dtype: int64

In [15]:
bios['born_country'].value_counts().tail(3)     # count of atheletes by country

born_country
Munich     1
Altmark    1
Aarhus     1
Name: count, dtype: int64

In [16]:
bios[['born_country', 'born_city']].value_counts().head(3)     # count of atheletes by 
                                                               # country, city

born_country  born_city      
HUN           Budapest           1378
RUS           Moskva (Moscow)     883
NOR           Oslo                708
Name: count, dtype: int64

In [17]:
print(f'{len(bios) = }')                                         # len of df
print(f'{bios.apply(lambda x: True, axis=1).sum().item() = }')   # len of df, very indirect way
print(f'{bios.duplicated().sum().item() = }')                    # len of duplicated rows
print(f'{(~bios.duplicated()).sum().item() = }')                 # len of non duplicated rows
print(f'{~bios.duplicated().sum().item() = }')                   # negation of the len of duplicated
len(bios.duplicated())


len(bios) = 145500
bios.apply(lambda x: True, axis=1).sum().item() = 145500
bios.duplicated().sum().item() = 0
(~bios.duplicated()).sum().item() = 145500
~bios.duplicated().sum().item() = -1


145500